# Multi-Layered Perceptron

Basically a feed-forward network, meaning data flow is forward.

**Note**: 

**We can't say backpropagation exists in perceptron training or MLP so it is feedback. Here, the data flows forward i.e input -> hidden -> output but backpropagation only feeds the previous layers, the gradients and not the output itself, unline Recurrent Neural Networks.**

Let the total layers in the network  be $\large{L}$.

For a single layer $\large{l}$:

- Number of neurons $= n_l$
- Weights matrix: $= \large{W^{(l)} \in \mathbf{R}^{n_l * n_{l-1}}}$
- Bias vector $\large{= b^{(l)} \in \mathbf{R}^{n_l}}$
- Input of layer $\large{l} = $ $\large{a^{(l - 1)}}$
- Output of layer $\large{l} = \large{a^{(l)}}$

The most confusinng idea might well be that, how does vectorization works here.

*Simple,*

Weights for a layer is a matrix

Inputs for a layer is a matrix

Bias for a layer is a vector

---

## Making backpropagation work with vectorization

- Let $\large{\mathcal{L}}$ be the loss function.
- $\large{\delta^{[l]} \equiv \frac{\partial{\mathcal{L}}}{\partial{z^{[l]}}}}$ be the error at layer $l$ aka the backpropagated gradient

---

**For the output layer $L$**

If the loss for that layer is: $\Large{\delta^{L} = \nabla_{a^{[L]}} \cdot \sigma\prime(z^{[L]})}$ basically partial derivative of loss with respect to the output of the layer mutliplied with the derivative of the activation function(sigmoid here).

Remember that the nabla operator yields,

$ \Large{(a^{[l]} - y)} $

For any other layer $l$,

$\large{\delta^{[l]} = (\mathbf{W}^{[l+1]})^T \delta^{[l+1]} \cdot \sigma \prime(z^{[l]})}$

$\large{\delta ^ {[l + 1]}}$ $\text{is the gradient from the layer \textbf{above}}$

Layer above means the layer from the previous backprop step.

---

**Now we compute the gradients with respect to the weights and biases for each layer**

$$ \Large{\frac{\partial{\mathcal{L}}}{\partial{W^{[l]}}}  = \delta ^{[l]} \cdot (A^{[l - 1]}) ^ T} $$ 
$$ \Large{\frac{\partial{\mathcal{L}}}{\partial{b^{[l]}}} = \sum\limits_{i = 1}^{B}(\delta_i^{l})}$$

Say for 1000 samples, our batch is 64, so the value $B$ is 64. i.e Batch size

In [2]:
import numpy as np
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# from keras.models import Sequential
# from keras.layers import Dense
# from keras.optimizers import Adam
import torch
import torch.nn as nn
import torch.optim as optim

### JUST A QUICK FUCK YOU TO TENSORFLOW. HONESTLY FUCK YOU.

In [11]:
X, y = make_classification(n_samples=1000, n_classes=2, n_informative = 10, n_features=14, random_state = 69, n_redundant = 4)

In [12]:
X_train, X_test, y_train, y_test = train_test_split(X,y, test_size=0.2, random_state=69)

In [13]:
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [14]:
X_train = torch.tensor(X_train, dtype=torch.float32)
X_test = torch.tensor(X_test, dtype=torch.float32)
y_train = torch.tensor(y_train, dtype=torch.float32).unsqueeze(1)
y_test = torch.tensor(y_test, dtype=torch.float32).unsqueeze(1)

# Basically add another dimension along the axis passed as parameter. So, just if it was (1000,) it will be (1,1000)

In [18]:
class MLP(nn.Module):
    def __init__(self, inputSize):
        super(MLP, self).__init__() ## Call the init method here
        self.net = nn.Sequential(nn.Linear(inputSize, 64), nn.ReLU(), nn.Linear(64, 32), nn.ReLU(), nn.Linear(32,1), nn.Sigmoid())

    def forward(self, X):
        return self.net(X)

model = MLP(inputSize = 14)
criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters(), lr = 0.001)

In [21]:
epochs = 2000
batch_size = 64

for epoch in range(epochs):
    permutation = torch.randperm(X_train.size(0))
    epoch_loss = 0

    for i in range(0, X_train.size(0), batch_size):
        indices = permutation[i:i+batch_size]
        batch_X, batch_y = X_train[indices], y_train[indices]

        optimizer.zero_grad()
        outputs = model(batch_X)
        loss = criterion(outputs, batch_y)
        loss.backward()
        optimizer.step()

        epoch_loss += loss.item()
    
    print(f"Epoch = {(epoch + 1)/epochs}, Loss = {epoch_loss}")

Epoch = 0.0005, Loss = 2.6069272309541702
Epoch = 0.001, Loss = 2.5867713689804077
Epoch = 0.0015, Loss = 2.4565906822681427
Epoch = 0.002, Loss = 2.400694914162159
Epoch = 0.0025, Loss = 2.2976574897766113
Epoch = 0.003, Loss = 2.2034079283475876
Epoch = 0.0035, Loss = 2.1619491204619408
Epoch = 0.004, Loss = 2.116922415792942
Epoch = 0.0045, Loss = 1.971338078379631
Epoch = 0.005, Loss = 1.9582265093922615
Epoch = 0.0055, Loss = 1.9175040647387505
Epoch = 0.006, Loss = 1.854979895055294
Epoch = 0.0065, Loss = 1.7791800871491432
Epoch = 0.007, Loss = 1.8357966393232346
Epoch = 0.0075, Loss = 1.6826174780726433
Epoch = 0.008, Loss = 1.6500143855810165
Epoch = 0.0085, Loss = 1.6004959046840668
Epoch = 0.009, Loss = 1.5432239696383476
Epoch = 0.0095, Loss = 1.4939719028770924
Epoch = 0.01, Loss = 1.4753999523818493
Epoch = 0.0105, Loss = 1.4981731548905373
Epoch = 0.011, Loss = 1.4119575396180153
Epoch = 0.0115, Loss = 1.4692822322249413
Epoch = 0.012, Loss = 1.3512770012021065
Epoch = 0

In [22]:
# Next commit accuracy. But gotta move on to CNNs